# Irodori-TTS 動作検証
**ランタイム設定**: ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選択してから実行してください。

- GitHub: https://github.com/Aratako/Irodori-TTS
- モデル: `Aratako/Irodori-TTS-500M`（HuggingFace から自動ダウンロード）
- 特徴: **絵文字による感情・スタイル制御**・参照音声によるボイスクローン
- 言語: **日本語専用**

## 1. GPU確認

In [ ]:
import torch

print(f"CUDA利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError("GPUが検出されません。ランタイムをT4 GPUに変更してください。")

## 2. ライブラリのインストール
初回は依存ライブラリのダウンロードが発生します（数分かかる場合があります）。

In [ ]:
import os

# リポジトリのクローン（既に存在する場合はスキップ）
if not os.path.exists("Irodori-TTS"):
    !git clone https://github.com/Aratako/Irodori-TTS.git
else:
    print("Irodori-TTS ディレクトリが既に存在します。スキップ。")

# dacvae は pyproject.toml で uv 専用の GitHub ソースが指定されており
# 通常の pip install -e . では自動インストールされないため個別に追加する
!pip install -q git+https://github.com/facebookresearch/dacvae.git

%cd Irodori-TTS
!pip install -e . -q

In [ ]:
import importlib
import irodori_tts
print(f"irodori-tts: OK")

from irodori_tts.inference_runtime import (
    InferenceRuntime,
    RuntimeKey,
    SamplingRequest,
    save_wav,
)
print("インポート完了！")

## 3. モデルロード
初回は約 2GB のダウンロードが発生します（HuggingFace より自動取得）。

In [ ]:
import torch
from huggingface_hub import hf_hub_download
from irodori_tts.inference_runtime import InferenceRuntime, RuntimeKey

MODEL_REPO = "Aratako/Irodori-TTS-500M"
# より高品質な 2.5B モデルを使う場合:
# MODEL_REPO = "Aratako/Irodori-TTS-2.5B"

print(f"モデルのダウンロード中: {MODEL_REPO}")
checkpoint_path = hf_hub_download(repo_id=MODEL_REPO, filename="model.safetensors")
print(f"チェックポイント: {checkpoint_path}")

# model_precision と codec_precision は必ず同じにすること。
# 異なると linear 層で dtype ミスマッチが発生する。
# T4 (15GB VRAM) なら fp32 でも 500M モデルは問題なく動作する。
PRECISION = "fp32"

runtime = InferenceRuntime.from_key(
    RuntimeKey(
        checkpoint=checkpoint_path,
        model_device="cuda",
        codec_repo="facebook/dacvae-watermarked",
        model_precision=PRECISION,
        codec_device="cuda",
        codec_precision=PRECISION,
        enable_watermark=False,
        compile_model=False,
        compile_dynamic=False,
    )
)
print("ロード完了！")
print(f"VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB 使用中")

## 4. 参照音声のアップロード（ボイスクローン用）

アップロードした音声の声質を再現します。
- **3〜30秒**のクリアな WAV ファイルを用意してください
- ノイズ・BGMなし推奨
- **ASMR ささやき音声をアップロードすると、そのままASMR声で生成されます**

参照音声なしで生成する場合は、このセルをスキップして **セル 6（no_ref モード）** を実行してください。

In [ ]:
from google.colab import files
import IPython.display as ipd

print("WAVファイルを選択してアップロードしてください...")
uploaded = files.upload()

if not uploaded:
    raise ValueError("ファイルがアップロードされませんでした。")

REF_AUDIO_PATH = list(uploaded.keys())[0]
print(f"アップロード完了: {REF_AUDIO_PATH}")

print("参照音声の確認:")
ipd.display(ipd.Audio(REF_AUDIO_PATH))

## 5. ASMRバッチ生成（ボイスクローンあり）

アップロードした参照音声の声質を再現してASMRを生成します。

### 絵文字スタイル制御
テキストに絵文字を埋め込むことで感情・トーンを制御できます:

| 絵文字 | 効果 |
|--------|------|
| 😌 | 穏やか・落ち着き |
| 😊 | 優しい・明るい |
| 😭 | 感情的・切ない |
| 🎵 | 歌うような |
| 💤 | 眠そう・まったり |

In [ ]:
import soundfile as sf
import IPython.display as ipd
import gc, torch, glob, os, re
from irodori_tts.inference_runtime import SamplingRequest, save_wav

# ============================================================
# ★ ASMRスクリプト（絵文字でニュアンスを制御）
#    漢字より ひらがな・カタカナ の方が発音精度が高い
# ============================================================
TEXTS = [
    "ねえ……きこえてる？そっと、みみをすませて……😌",
    "きょうも、おつかれさまでした……ゆっくり、いきをはいて。だいじょうぶですよ。😌",
    "もうすこしだけ、そばにいますね……なにも、しんぱいしなくていいです。😌",
    "めをとじて……ただ、わたしのこえだけ、きいていて。💤",
    "おやすみなさい……ゆっくり、ねむれますように……。💤",
]

# ---- 前回の出力ファイルを削除 & GPU メモリ確認 ----
for old in glob.glob("irodori_*.wav"):
    os.remove(old)
    print(f"削除: {old}")
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB 使用中")

def safe_filename(prefix, text, seed, max_len=30):
    name = re.sub(r'[\\/:*?"<>|]', "", text)
    name = re.sub(r"[😌😊😭🎵💤\s]+", "_", name.strip())
    return f"{prefix}_seed{seed}_{name[:max_len]}.wav"

# ============================================================
# ★ 生成パラメータ（ASMR向けチューニング済み）
# ============================================================
NUM_STEPS = 60          # 拡散ステップ数（多いほど高品質・低速）
CFG_SCALE_TEXT = 2.0    # テキスト CFG（低いほど柔らかい声）
CFG_SCALE_SPEAKER = 4.0 # スピーカー CFG（低いほど参照音声への拘束が弱い）

# ============================================================
# ★ シード値
#    None    → ランダム（毎回異なる音声）
#    整数値  → 固定（同じ seed なら同じ音声が再現される）
#    例: SEED = 42
# ============================================================
SEED = None

print(f"ニュアンス: クローンあり | seed: {SEED if SEED is not None else 'ランダム'}")
print("音声生成中...")
output_files = []

for i, text in enumerate(TEXTS):
    print(f"\n[{i+1}/{len(TEXTS)}] 生成中: {text}")
    result = runtime.synthesize(
        SamplingRequest(
            text=text,
            ref_wav=REF_AUDIO_PATH,
            no_ref=False,
            num_steps=NUM_STEPS,
            cfg_scale_text=CFG_SCALE_TEXT,
            cfg_scale_speaker=CFG_SCALE_SPEAKER,
            cfg_guidance_mode="independent",
            trim_tail=True,
            seed=SEED,
        )
    )
    path = safe_filename("irodori_clone", text, result.used_seed)
    save_wav(path, result.audio, result.sample_rate)
    output_files.append(path)
    print(f"保存: {path}  (seed: {result.used_seed})")
    ipd.display(ipd.Audio(path))

print(f"\nバッチ生成完了！ {len(output_files)}件")

## 6. ASMRバッチ生成（参照音声なし・no_ref モード）

参照音声なしでモデルが自然に選んだ声質で生成します。
セル 4・5 をスキップしてここから実行することもできます。

In [ ]:
import soundfile as sf
import IPython.display as ipd
import gc, torch, glob, os, re
from irodori_tts.inference_runtime import SamplingRequest, save_wav

# ============================================================
# ★ ASMRスクリプト（絵文字でニュアンスを制御）
# ============================================================
TEXTS = [
    "ねえ……きこえてる？そっと、みみをすませて……😌",
    "きょうも、おつかれさまでした……ゆっくり、いきをはいて。だいじょうぶですよ。😌",
    "もうすこしだけ、そばにいますね……なにも、しんぱいしなくていいです。😌",
    "めをとじて……ただ、わたしのこえだけ、きいていて。💤",
    "おやすみなさい……ゆっくり、ねむれますように……。💤",
]

# ---- 前回の出力ファイルを削除 & GPU メモリ確認 ----
for old in glob.glob("irodori_noref_*.wav"):
    os.remove(old)
    print(f"削除: {old}")
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB 使用中")

def safe_filename_noref(text, seed, max_len=30):
    name = re.sub(r'[\\/:*?"<>|]', "", text)
    name = re.sub(r"[😌😊😭🎵💤\s]+", "_", name.strip())
    return f"irodori_noref_seed{seed}_{name[:max_len]}.wav"

# ============================================================
# ★ 生成パラメータ
# ============================================================
NUM_STEPS = 60
CFG_SCALE_TEXT = 3.0    # no_ref 時はやや高めが安定

# ============================================================
# ★ シード値
#    None    → ランダム（毎回異なる音声）
#    整数値  → 固定（同じ seed なら同じ音声が再現される）
#    例: SEED = 42
# ============================================================
SEED = None

print(f"ニュアンス: no_ref | seed: {SEED if SEED is not None else 'ランダム'}")
print("音声生成中（参照音声なし）...")
output_files = []

for i, text in enumerate(TEXTS):
    print(f"\n[{i+1}/{len(TEXTS)}] 生成中: {text}")
    result = runtime.synthesize(
        SamplingRequest(
            text=text,
            no_ref=True,
            num_steps=NUM_STEPS,
            cfg_scale_text=CFG_SCALE_TEXT,
            cfg_scale_speaker=0.0,
            cfg_guidance_mode="joint",
            trim_tail=True,
            seed=SEED,
        )
    )
    path = safe_filename_noref(text, result.used_seed)
    save_wav(path, result.audio, result.sample_rate)
    output_files.append(path)
    print(f"保存: {path}  (seed: {result.used_seed})")
    ipd.display(ipd.Audio(path))

print(f"\nバッチ生成完了！ {len(output_files)}件")

## 7. 生成ファイルのダウンロード

In [ ]:
from google.colab import files
import glob, os

exclude = set()
if 'REF_AUDIO_PATH' in dir():
    exclude.add(os.path.basename(REF_AUDIO_PATH))

wav_files = [
    f for f in glob.glob("irodori_*.wav")
    if os.path.basename(f) not in exclude
]
print(f"ダウンロード対象: {len(wav_files)}件")

for f in wav_files:
    print(f"ダウンロード: {f}")
    files.download(f)